# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# Load the raw aviation accident data and inspect its structure.
from pathlib import Path

DATA_PATH = Path("data/AviationData.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/AviationData.csv")

df = pd.read_csv(DATA_PATH, encoding="latin1", low_memory=False)
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(15))
display(df.describe(include="all").T.head(20))

Shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

None

,missing_count
Schedule,76307
Air.carrier,72241
FAR.Description,56866
Aircraft.Category,56602
Longitude,54516
Latitude,54507
Airport.Code,38757
Airport.Name,36185
Broad.phase.of.flight,27165
Publication.Date,13771


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [4]:
# Focus on accident records involving airplanes. Aircraft.Category is missing for many older records, so we do not infer "Airplane" where
# it is unknown: that would risk mixing aircraft types. Incident-only records are also excluded.

df["Investigation.Type"] = df["Investigation.Type"].astype("string").str.strip()
df["Aircraft.Category"] = df["Aircraft.Category"].astype("string").str.strip()
df = df.loc[
    df["Investigation.Type"].str.casefold().eq("accident")
    & df["Aircraft.Category"].str.casefold().eq("airplane")
].copy()

# Parse dates and coerce numeric fields; invalid entries become NaN.
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
numeric_cols = [
    "Number.of.Engines", "Total.Fatal.Injuries", "Total.Serious.Injuries",
    "Total.Minor.Injuries", "Total.Uninjured"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Filtered airplane accidents:", df.shape)
display(df[["Event.Date", "Make", "Model", "Aircraft.Category"]].head())

Filtered airplane accidents: (25963, 31)


,Event.Date,Make,Model,Aircraft.Category
5,1979-09-17,Mcdonnell Douglas,DC9,Airplane
7,1982-01-01,Cessna,140,Airplane
8,1982-01-01,Cessna,401B,Airplane
12,1982-01-02,Bellanca,17-30A,Airplane
13,1982-01-02,Cessna,R172K,Airplane


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [5]:
# Construct occupant and injury metrics. Missing injury counts are not automatically zero-filled because missing is not evidence that no
# one was injured. 
# For the occupant denominator, use records with at least one reported occupant count and calculate the severe-injury fraction only where 
# the denominator is positive.

injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries",
               "Total.Minor.Injuries", "Total.Uninjured"]
df["Total.Occupants.Reported"] = df[injury_cols].sum(axis=1, min_count=1)
df["Fatal.Serious.Count"] = df[["Total.Fatal.Injuries", "Total.Serious.Injuries"]].sum(axis=1, min_count=2)
df["Fatal.Serious.Fraction"] = (
    df["Fatal.Serious.Count"] / df["Total.Occupants.Reported"]
).where(df["Total.Occupants.Reported"] > 0)

# Retain the individual injury counts and report how much data supports this metric.
print("Rows with a usable occupant denominator:", df["Total.Occupants.Reported"].notna().sum())
print("Rows with a calculable severe-injury fraction:", df["Fatal.Serious.Fraction"].notna().sum())
display(df[["Total.Occupants.Reported", "Fatal.Serious.Count", "Fatal.Serious.Fraction"]].describe())

Rows with a usable occupant denominator: 25940
Rows with a calculable severe-injury fraction: 22421


,Total.Occupants.Reported,Fatal.Serious.Count,Fatal.Serious.Fraction
count,25940.000000,22667.000000,22421.000000
mean,5.155243,0.931707,0.328829
std,25.202770,6.218755,0.450309
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,1.000000
max,576.000000,295.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [6]:
# Standardize aircraft damage labels and create a binary destruction outcome.

df["Aircraft.damage"] = df["Aircraft.damage"].astype("string").str.strip().str.title()
damage_map = {
    "Destroyed": "Destroyed",
    "Substantial": "Substantial",
    "Minor": "Minor",
    "Unknown": "Unknown",
    "Nan": pd.NA,
    "": pd.NA
}
df["Aircraft.damage"] = df["Aircraft.damage"].replace(damage_map)
df["Aircraft.Destroyed"] = df["Aircraft.damage"].map({"Destroyed": 1, "Substantial": 0, "Minor": 0}).astype(float)
display(df["Aircraft.damage"].value_counts(dropna=False).to_frame("count"))
display(df["Aircraft.Destroyed"].value_counts(dropna=False).to_frame("count"))

,count
Aircraft.damage,
Substantial,21694
Destroyed,3552
<NA>,458
Minor,181
Unknown,78


,count
Aircraft.Destroyed,
0.0,21875
1.0,3552
NaN,536


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [7]:
# Normalize Make text: trim whitespace, collapse repeated spaces, and title-case.
# Exclude blank/unknown make labels and retain makes with at least 50 records so comparisons are not dominated by extremely small samples.

df["Make"] = df["Make"].astype("string").str.strip().str.replace(r"\\s+", " ", regex=True)
df["Make"] = df["Make"].str.title()
unknown_makes = {"", "Unknown", "Unk", "N/A", "Na", "None", "<Na>"}
df.loc[df["Make"].isin(unknown_makes), "Make"] = pd.NA
make_counts = df["Make"].value_counts()
eligible_makes = make_counts[make_counts >= 50].index
df = df[df["Make"].isin(eligible_makes)].copy()
print("Distinct eligible makes:", df["Make"].nunique())
display(df["Make"].value_counts().head(20).to_frame("accidents"))

Distinct eligible makes: 37


,accidents
Make,
Cessna,8336
Piper,4642
Beech,1637
Boeing,589
Mooney,413
Bellanca,280
Grumman,249
Maule,232
Aeronca,227


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [8]:
# Remove records without a usable model; standardize model strings and create a make/model key because the same model label can be used
# by different manufacturers.

df["Model"] = df["Model"].astype("string").str.strip().str.upper().str.replace(r"\\s+", " ", regex=True)
df.loc[df["Model"].isin(["", "UNKNOWN", "UNK", "N/A", "NA", "NONE", "<NA>"]), "Model"] = pd.NA
df = df.dropna(subset=["Model"]).copy()
df["Make.Model"] = df["Make"].astype(str) + " | " + df["Model"].astype(str)

# Inspect repeated model labels across manufacturers.
model_make_counts = df.groupby("Model")["Make"].nunique().sort_values(ascending=False)
print("Model labels used by multiple makes:", int((model_make_counts > 1).sum()))
display(model_make_counts[model_make_counts > 1].head(10).to_frame("number_of_makes"))

Model labels used by multiple makes: 105


,number_of_makes
Model,
S2R,4
7EC,3
7GCAA,3
8GCBC,3
7GCBC,3
7ECA,3
8KCAB,3
500,3
7AC,3


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [9]:
# Clean selected explanatory variables. Preserve missing values as missing and map common placeholders to NA; standardize spacing/case 
# for consistent grouping.

categorical_cols = [
    "Engine.Type", "Weather.Condition", "Purpose.of.flight",
    "Broad.phase.of.flight", "Amateur.Built"
]
for col in categorical_cols:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\\s+", " ", regex=True)
    df[col] = df[col].replace({
        "": pd.NA, "UNK": pd.NA, "Unknown": pd.NA, "UNKNOWN": pd.NA,
        "N/A": pd.NA, "NA": pd.NA, "None": pd.NA
    })

# Standardize common category variants.
df["Weather.Condition"] = df["Weather.Condition"].str.upper()
df["Purpose.of.flight"] = df["Purpose.of.flight"].str.title()
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].str.title()
df["Engine.Type"] = df["Engine.Type"].str.title()
df["Number.of.Engines"] = df["Number.of.Engines"].where(df["Number.of.Engines"].between(0, 8))
for col in categorical_cols:
    print(f"\\n{col}: {df[col].nunique(dropna=True)} categories; {df[col].isna().sum()} missing")
    display(df[col].value_counts(dropna=False).head(12).to_frame("count"))

\nEngine.Type: 5 categories; 2591 missing


,count
Engine.Type,
Reciprocating,15534
<NA>,2591
Turbo Prop,919
Turbo Fan,377
Turbo Jet,51
Turbo Shaft,8


\nWeather.Condition: 3 categories; 1640 missing


,count
Weather.Condition,
VMC,16557
<NA>,1640
IMC,1135
UNK,148


\nPurpose.of.flight: 23 categories; 2554 missing


,count
Purpose.of.flight,
Personal,11479
Instructional,2749
<NA>,2554
Aerial Application,844
Business,650
Positioning,254
Skydiving,158
Aerial Observation,155
Ferry,120


\nBroad.phase.of.flight: 11 categories; 14402 missing


,count
Broad.phase.of.flight,
<NA>,14402
Landing,1872
Takeoff,977
Cruise,690
Approach,494
Maneuvering,397
Taxi,197
Descent,136
Go-Around,135


\nAmateur.Built: 2 categories; 6 missing


,count
Amateur.Built,
No,19269
Yes,205
<NA>,6


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [10]:
# Drop columns with very high missingness or identifiers/text that are not needed for the requested make/model, damage, injury, and 
# factor analyses. Keep the original event date and key analytical fields for future extensions.

missing_fraction = df.isna().mean().sort_values(ascending=False)
display(missing_fraction.to_frame("missing_fraction").head(15))

required_analysis_cols = {
    "Event.Id", "Event.Date", "Make", "Model", "Make.Model",
    "Aircraft.damage", "Aircraft.Destroyed", "Fatal.Serious.Fraction",
    "Total.Occupants.Reported", "Engine.Type", "Weather.Condition",
    "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"
}
drop_cols = [
    col for col in df.columns
    if df[col].isna().mean() > 0.70 and col not in required_analysis_cols
]
# High-cardinality identifiers and narrative/reporting fields add little to this analysis.
drop_cols += [c for c in [
    "Accident.Number", "Registration.Number", "Airport.Code", "Airport.Name",
    "Air.carrier", "Report.Status", "Publication.Date", "FAR.Description",
    "Latitude", "Longitude", "Location", "Country", "Schedule"
] if c in df.columns and c not in drop_cols]
df.drop(columns=drop_cols, inplace=True, errors="ignore")
print("Dropped columns:", drop_cols)
print("Final cleaned shape:", df.shape)
display(df.head())

,missing_fraction
Schedule,0.913244
Broad.phase.of.flight,0.739322
Air.carrier,0.602977
Airport.Code,0.352669
Airport.Name,0.329415
Longitude,0.195534
Latitude,0.195380
Report.Status,0.155082
Fatal.Serious.Fraction,0.140298
Engine.Type,0.133008


Dropped columns: ['Schedule', 'Accident.Number', 'Registration.Number', 'Airport.Code', 'Airport.Name', 'Air.carrier', 'Report.Status', 'Publication.Date', 'FAR.Description', 'Latitude', 'Longitude', 'Location', 'Country']
Final cleaned shape: (19480, 23)


,Event.Id,Investigation.Type,Event.Date,Injury.Severity,Aircraft.damage,Aircraft.Category,Make,Model,Amateur.Built,Number.of.Engines,...,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Total.Occupants.Reported,Fatal.Serious.Count,Fatal.Serious.Fraction,Aircraft.Destroyed,Make.Model
5,20170710X52551,Accident,1979-09-17,Non-Fatal,Substantial,Airplane,Mcdonnell Douglas,DC9,No,2.0,...,NaN,1.0,44.0,VMC,Climb,45.0,NaN,NaN,0.0,Mcdonnell Douglas | DC9
7,20020909X01562,Accident,1982-01-01,Non-Fatal,Substantial,Airplane,Cessna,140,No,1.0,...,0.0,0.0,2.0,VMC,Takeoff,2.0,0.0,0.0,0.0,Cessna | 140
8,20020909X01561,Accident,1982-01-01,Non-Fatal,Substantial,Airplane,Cessna,401B,No,2.0,...,0.0,0.0,2.0,IMC,Landing,2.0,0.0,0.0,0.0,Cessna | 401B
12,20020917X02148,Accident,1982-01-02,Non-Fatal,Destroyed,Airplane,Bellanca,17-30A,No,1.0,...,0.0,1.0,0.0,IMC,Cruise,1.0,0.0,0.0,1.0,Bellanca | 17-30A
13,20020917X02134,Accident,1982-01-02,Fatal(1),Destroyed,Airplane,Cessna,R172K,No,1.0,...,0.0,0.0,0.0,IMC,Takeoff,1.0,1.0,1.0,1.0,Cessna | R172K


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [11]:
# Save a reusable cleaned dataset for the analysis notebook.

OUTPUT_PATH = Path("data/aviation_accidents_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df):,} cleaned records to {OUTPUT_PATH}")

Saved 19,480 cleaned records to data\aviation_accidents_cleaned.csv
